# Explore TensorFlow SavedModel

##### Model

In [1]:
import pathlib
import tensorflow as tf

model_version = "0001"
model_name = "mnist_model"
model_path = pathlib.Path(model_name, model_version)

model_saved = tf.saved_model.load(model_path)
model_saved


2023-10-16 07:44:25.882776: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-10-16 07:44:26.166263: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-10-16 07:44:26.168602: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-10-16 07:44:27.226519: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2023-10-16 07:44:28.645275: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Do

<tensorflow.python.saved_model.load.Loader._recreate_base_user_object.<locals>._UserObject at 0x7f07286c5490>

In [2]:
hasattr(model_saved, "predict")

False

##### Data

In [3]:
import tensorflow as tf

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
print(f"x_train: {x_train.shape}")
print(f"y_test: {y_test.shape}")

x_train: (60000, 28, 28)
y_test: (10000,)


In [4]:
import pandas as pd

x_train, x_test = x_train / 255.0, x_test / 255.0
pd.Series(x_test.ravel()).describe()

count    7.840000e+06
mean     1.325146e-01
std      3.104803e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.000000e+00
dtype: float64

In [5]:
import numpy as np

x_train = x_train.astype(np.float32)
x_test = x_test.astype(np.float32)
y_train = y_train.astype(int)
y_test = y_test.astype(int)

##### Test Model

In [6]:
model_saved(tf.constant(x_test), training=False)

ValueError: Could not find matching concrete function to call loaded from the SavedModel. Got:
  Positional arguments (3 total):
    * <tf.Tensor 'inputs:0' shape=(10000, 28, 28) dtype=float32>
    * False
    * None
  Keyword arguments: {}

 Expected these arguments to match one of the following 2 option(s):

Option 1:
  Positional arguments (3 total):
    * TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='flatten_input')
    * True
    * None
  Keyword arguments: {}

Option 2:
  Positional arguments (3 total):
    * TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='flatten_input')
    * False
    * None
  Keyword arguments: {}

In [ ]:
input = tf.keras.layers.Input(shape=(28, 28))
output = model_saved(input, training=False)
model = tf.keras.models.Model(input=[input], output=[output])
model

TypeError: You are passing KerasTensor(type_spec=TensorSpec(shape=(None, 28, 28), dtype=tf.float32, name='input_2'), name='input_2', description="created by layer 'input_2'"), an intermediate Keras symbolic input/output, to a TF API that does not allow registering custom dispatchers, such as `tf.cond`, `tf.function`, gradient tapes, or `tf.map_fn`. Keras Functional model construction only supports TF API calls that *do* support dispatching, such as `tf.math.add` or `tf.reshape`. Other APIs cannot be called directly on symbolic Kerasinputs/outputs. You can work around this limitation by putting the operation in a custom Keras layer `call` and calling that layer on this symbolic input/output.

In [9]:
!export ML_PATH="/home/yuncong/Projects/data-science/homl/ch19_training_and_deploying_tensorflow_models_at_scale"
!saved_model_cli show --dir "mnist_model/0001" --all

2023-10-16 07:49:57.818215: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-10-16 07:49:57.854469: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-10-16 07:49:57.854857: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-10-16 07:49:58.818947: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2023-10-16 07:49:59.874790: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Do

In [17]:
file_path = "x_test_0_to_3.npy"
np.save(file_path, x_test[:3, :, :, np.newaxis])

In [18]:
!saved_model_cli run --dir "mnist_model/0001" --tag_set serve --signature_def serving_default --inputs flatten_input="x_test_0_to_3.npy"

2023-10-16 08:03:25.240255: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-10-16 08:03:25.279430: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-10-16 08:03:25.279815: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-10-16 08:03:26.265894: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2023-10-16 08:03:27.342523: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Do